In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
from scipy.sparse.linalg import norm
import nltk
from nltk.corpus import stopwords
import re
from html import unescape
import gzip

# Download the stopwords package from NLTK
nltk.download('stopwords')

#Read and unzip the json file, uses yiel instead of return to read line by line
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

#Create the pandas DF
def getDF(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

#Remove all HTML labels, tags, and special characters to the Title
def clean_songinfo(songinfo):
    songinfo = unescape(songinfo)
    songinfo = re.sub(r'<[^>]+>', '', songinfo)
    songinfo = re.sub(r'&[^;]+;', '', songinfo)
    return songinfo

#Load and clean data
file_path = 'meta_Digital_Music.json.gz'
songs = getDF(file_path)
songs['title'] = songs['title'].apply(clean_songinfo)
songs['brand'] = songs['brand'].apply(clean_songinfo) if 'brand' in songs.columns else ""
songs['main_cat'] = songs['main_cat'].apply(clean_songinfo)if 'main_cat' in songs.columns else ""

songs['combined_info'] = songs['title'] + ' ' + songs['brand'] + ' ' + songs['main_cat']

#Vectorize using TF IDF, removing stop-words in different idioms, limit the dimensionality and normalizing
tfidf_vectorizer = TfidfVectorizer(
    stop_words=stopwords.words('english') + stopwords.words('spanish')
    + stopwords.words('french') + stopwords.words('portuguese')
    + stopwords.words('german'),
    max_features=5000,
    norm='l2'
)
tfidf_matrix = tfidf_vectorizer.fit_transform(songs['combined_info'])


#calculate the similarity with dot product of every song
def efficient_dot_product(matrix, batch_size):
    dot_sim = []
    for i in range(0, matrix.shape[0], batch_size):
        start_index = i
        end_index = min(i + batch_size, matrix.shape[0])
        batch_dot_sim = matrix[start_index:end_index].dot(matrix.T)
        dot_sim.append(batch_dot_sim)
    # Concatenate the sparse matrices vertically
    return sparse.vstack(dot_sim, format='csr')

batch_size = 500

dot_sim_matrix = efficient_dot_product(tfidf_matrix, batch_size)

#Save the title and its index
titles = songs['title']
indices = pd.Series(songs.index, index=songs['title'])


#get the similar recomendations
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(dot_sim_matrix[idx].toarray()[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]  # Get top 10 recommendations
    song_indices = [i[0] for i in sim_scores]
    return titles.iloc[song_indices]

#User interaction to ask for a song until the user writes the key word 'exit' or 'quit'
while True:
    song_title = input("Enter a song title or 'exit' to quit: ")
    if song_title == 'exit':
        break
    if song_title in titles.values:
        print("Top 10 recommendations for '{}':".format(song_title))
        recommended_titles = get_recommendations(song_title)
        for rec_title in recommended_titles:
            print(rec_title)
    else:
        print("We don't have recommendations for '{}'.".format(song_title))


"""
Sample of songs to test:

Magical
Vamos A Bailar: Salsa Merengue Conga
Die Schopefung
Autres Voix Autres Blues
Edyta Gorniak

"""


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Enter a song title or 'exit' to quit: Edyta Gorniak
Top 10 recommendations for 'Edyta Gorniak':
Hymns Collection: Hymns 1 & 2
Early Works - Don Francisco
So You Wanna Go Back to Egypt
Early Works - Dallas Holm
The Glory of Love (Solo Piano)
Songs for Worship: Volumes 1 and 2
The Life: The Complete Recorded Trilogy on the Life of Christ
The Lord's Supper / Be Exalted
Sing & Play Swamp Stomp Music
The Music Connection Grade 3, CD 7
Enter a song title or 'exit' to quit: exit
